# Note

This notebook needs the following zip file containing the GraphBlas backendto run.

[gbtl_src.zip](https://github.com/mattnkrueger/gbtl_src) (link to github page)

--

**Environments**
- `CPU`, Python3 (GBlas & OpenMP)
- `T4 GPU`, Python3 (CUDA)


In [ ]:
# unzip gbtl repo
!unzip -oq gbtl_src.zip -d /content/
!g++ -std=c++17 -O3 \
    -I /content/gbtl_src \
    -I /content/gbtl_src/graphblas/platforms/optimized_sequential \
    apsp_delta_step_gbtl.cpp -o apsp_delta_step_gbtl

## 1) Sequential GBlas - `apsp_delta_step_gbtl.cpp`

In [ ]:
%%writefile apsp_delta_step_gbtl.cpp
#include <chrono>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <iostream>
#include <vector>

#include <graphblas/graphblas.hpp>
#include <algorithms/sssp.hpp>

#define INF 1000000000
using Weight = unsigned int;

/* ================= GRAPH ================= */
typedef struct {
    int vertices;
    int **adj_matrix;
} Graph;

Graph* graph_create(int vertices) {
    Graph *g = (Graph *)malloc(sizeof(Graph));
    g->vertices = vertices;
    g->adj_matrix = (int **)malloc(vertices * sizeof(int *));
    for (int i = 0; i < vertices; i++) {
        g->adj_matrix[i] = (int *)malloc(vertices * sizeof(int));
        for (int j = 0; j < vertices; j++)
            g->adj_matrix[i][j] = (i == j) ? 0 : INF;
    }
    return g;
}

void graph_add_edge(Graph *g, int src, int dest, int weight) {
    if (src != dest && g->adj_matrix[src][dest] > weight)
        g->adj_matrix[src][dest] = weight;
}

void graph_free(Graph *g) {
    for (int i = 0; i < g->vertices; i++) free(g->adj_matrix[i]);
    free(g->adj_matrix);
    free(g);
}

void fill_random_edges(Graph *g, int density_pct, unsigned int seed) {
    srand(seed);
    int n = g->vertices;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
            if (i != j && (rand() % 100) < density_pct)
                graph_add_edge(g, i, j, (rand() % 100) + 1);
}

static int count_directed_edges(Graph *g) {
    int n = g->vertices, m = 0;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
            if (i != j && g->adj_matrix[i][j] < INF) m++;
    return m;
}

void matrix_free(int **m, int n) {
    for (int i = 0; i < n; i++) free(m[i]);
    free(m);
}

int main()
{
    int n;
    unsigned int delta_in;

    std::printf("=======================================================\n");
    std::printf("  DELTA-STEPPING APSP (Sequential GBlas / GBTL)\n");
    std::printf("=======================================================\n\n");

    std::printf("Enter number of nodes: ");
    if (std::scanf("%d", &n) != 1 || n < 2) {
        std::printf("Invalid number of nodes (must be >= 2).\n");
        return 1;
    }
    std::printf("Enter the Delta Value: ");
    if (std::scanf("%u", &delta_in) != 1 || delta_in == 0u) {
        std::printf("Delta must be > 0.\n"); return 1;
    }
    Weight delta = (Weight)delta_in;

    int density = (n <= 100) ? 40 : 20;
    std::printf("\n--- Building random graph ---\n");
    std::printf("Nodes: %d   Density: %d%%   Seed: 42\n", n, density);

    Graph *g = graph_create(n);
    fill_random_edges(g, density, 42);
    int m = count_directed_edges(g);

    std::printf("Graph generation complete.\n");
    std::printf("Nodes: %d\n", n);
    std::printf("Directed edges stored: %d\n", m);
    std::printf("Undirected edges stored: %d\n\n", m / 2);

    /* Build grb::Matrix once from adjacency */
    std::vector<grb::IndexType> rows, cols;
    std::vector<Weight>         vals;
    rows.reserve(m); cols.reserve(m); vals.reserve(m);
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
            if (i != j && g->adj_matrix[i][j] < INF) {
                rows.push_back((grb::IndexType)i);
                cols.push_back((grb::IndexType)j);
                vals.push_back((Weight)g->adj_matrix[i][j]);
            }

    grb::Matrix<Weight> A((grb::IndexType)n, (grb::IndexType)n);
    A.build(rows.begin(), cols.begin(), vals.begin(),
            (grb::IndexType)rows.size(), grb::Min<Weight>());

    /* APSP distance matrix (INF = unreachable, 0 on diagonal) */
    int **dist_ds = (int **)malloc(n * sizeof(int *));
    for (int i = 0; i < n; i++) {
        dist_ds[i] = (int *)malloc(n * sizeof(int));
        for (int j = 0; j < n; j++) dist_ds[i][j] = INF;
        dist_ds[i][i] = 0;
    }

    std::printf("Running Delta-Stepping APSP (Sequential GBlas)...\n");
    auto t0 = std::chrono::high_resolution_clock::now();
    for (int s = 0; s < n; s++) {
        grb::Vector<Weight> paths((grb::IndexType)n);
        algorithms::sssp_delta_step(A, delta, (grb::IndexType)s, paths);
        for (int v = 0; v < n; v++)
            if (paths.hasElement((grb::IndexType)v))
                dist_ds[s][v] = (int)paths.extractElement((grb::IndexType)v);
    }
    auto t1 = std::chrono::high_resolution_clock::now();
    double runtime_ms = std::chrono::duration<double>(t1 - t0).count() * 1000.0;

    std::printf("\n=======================================================\n");
    std::printf("       APSP RUNTIME (delta = %u)\n", delta_in);
    std::printf("=======================================================\n");
    std::printf(" %-32s | %-12s\n", "Algorithm", "Time (ms)");
    std::printf("-------------------------------------------------------\n");
    std::printf(" %-32s | %12.4f\n", "Delta-Stepping (Seq GBlas)", runtime_ms);

    matrix_free(dist_ds, n);
    graph_free(g);
    return 0;
}

Writing apsp_delta_step_gbtl.cpp


## 2) OpenMP - `apsp_delta_step_omp.cpp`

In [ ]:
%%writefile apsp_delta_step_omp.cpp
#include <chrono>
#include <climits>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <vector>

#include <omp.h>

#define INF 1000000000
using Weight = unsigned int;
static const Weight   INF_W      = 0xFFFFFFFFu;
static const uint32_t INF_BUCKET = 0xFFFFFFFFu;

/* ================= GRAPH ================= */
typedef struct {
    int vertices;
    int **adj_matrix;
} Graph;

Graph* graph_create(int vertices) {
    Graph *g = (Graph *)malloc(sizeof(Graph));
    g->vertices = vertices;
    g->adj_matrix = (int **)malloc(vertices * sizeof(int *));
    for (int i = 0; i < vertices; i++) {
        g->adj_matrix[i] = (int *)malloc(vertices * sizeof(int));
        for (int j = 0; j < vertices; j++)
            g->adj_matrix[i][j] = (i == j) ? 0 : INF;
    }
    return g;
}

void graph_add_edge(Graph *g, int s, int d, int w) {
    if (s != d && g->adj_matrix[s][d] > w) g->adj_matrix[s][d] = w;
}

void graph_free(Graph *g) {
    for (int i = 0; i < g->vertices; i++) free(g->adj_matrix[i]);
    free(g->adj_matrix);
    free(g);
}

void fill_random_edges(Graph *g, int density_pct, unsigned int seed) {
    srand(seed);
    int n = g->vertices;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
            if (i != j && (rand() % 100) < density_pct)
                graph_add_edge(g, i, j, (rand() % 100) + 1);
}

static int count_directed_edges(Graph *g) {
    int n = g->vertices, m = 0;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
            if (i != j && g->adj_matrix[i][j] < INF) m++;
    return m;
}

void matrix_free(int **m, int n) {
    for (int i = 0; i < n; i++) free(m[i]);
    free(m);
}

/* ================= CSR with light/heavy split ================= */
struct CSR {
    int n = 0;
    std::vector<long long> row_off;
    std::vector<long long> light_end;
    std::vector<int>       col_idx;
    std::vector<Weight>    weights;
};

static void build_csr_split(Graph *g, Weight delta, CSR &G) {
    int n = g->vertices;
    G.n = n;
    G.row_off.assign(n + 1, 0);
    G.light_end.assign(n, 0);
    long long m = 0;
    for (int u = 0; u < n; u++)
        for (int v = 0; v < n; v++)
            if (u != v && g->adj_matrix[u][v] < INF) m++;
    G.col_idx.resize(m);
    G.weights.resize(m);
    long long off = 0;
    for (int u = 0; u < n; u++) {
        G.row_off[u] = off;
        for (int v = 0; v < n; v++)
            if (u != v && g->adj_matrix[u][v] < INF) {
                Weight w = (Weight)g->adj_matrix[u][v];
                if (w <= delta) {
                    G.col_idx[off] = v;
                    G.weights[off] = w;
                    off++;
                }
            }
        G.light_end[u] = off;
        for (int v = 0; v < n; v++)
            if (u != v && g->adj_matrix[u][v] < INF) {
                Weight w = (Weight)g->adj_matrix[u][v];
                if (w > delta) {
                    G.col_idx[off] = v;
                    G.weights[off] = w;
                    off++;
                }
            }
    }
    G.row_off[n] = off;
}

/* Sequential delta-step (one per source; outer loop is OMP-parallel) */
static void seq_delta_step(int n, CSR const &G, int source, Weight delta,
                           std::vector<Weight> &tent)
{
    tent.assign(n, INF_W);
    std::vector<uint32_t> bidx(n, INF_BUCKET);
    tent[source] = 0;
    bidx[source] = 0;

    std::vector<char> in_R(n, 0);
    std::vector<int>  R, S;
    uint32_t cur_bucket = 0;

    while (true) {
        uint32_t min_b = INF_BUCKET;
        for (int v = 0; v < n; v++) {
            uint32_t b = bidx[v];
            if (b != INF_BUCKET && b >= cur_bucket && b < min_b) min_b = b;
        }
        if (min_b == INF_BUCKET) break;
        cur_bucket = min_b;

        std::fill(in_R.begin(), in_R.end(), 0);

        while (true) {
            S.clear();
            for (int v = 0; v < n; v++)
                if (bidx[v] == cur_bucket) S.push_back(v);
            if (S.empty()) break;

            for (int u : S) { in_R[u] = 1; bidx[u] = INF_BUCKET; }

            for (int u : S) {
                Weight tu = tent[u];
                if (tu == INF_W) continue;
                long long s = G.row_off[u];
                long long e = G.light_end[u];
                for (long long k = s; k < e; k++) {
                    int    w_v = G.col_idx[k];
                    Weight c   = G.weights[k];
                    Weight nd  = tu + c;
                    if (nd < tu) continue;
                    if (nd < tent[w_v]) {
                        tent[w_v] = nd;
                        bidx[w_v] = (uint32_t)(nd / delta);
                    }
                }
            }
        }

        R.clear();
        for (int v = 0; v < n; v++) if (in_R[v]) R.push_back(v);
        for (int u : R) {
            Weight tu = tent[u];
            if (tu == INF_W) continue;
            long long s = G.light_end[u];
            long long e = G.row_off[u + 1];
            for (long long k = s; k < e; k++) {
                int    w_v = G.col_idx[k];
                Weight c   = G.weights[k];
                Weight nd  = tu + c;
                if (nd < tu) continue;
                if (nd < tent[w_v]) {
                    tent[w_v] = nd;
                    bidx[w_v] = (uint32_t)(nd / delta);
                }
            }
        }

        cur_bucket++;
    }
}

int main()
{
    int n;
    unsigned int delta_in;

    std::printf("=======================================================\n");
    std::printf("  DELTA-STEPPING APSP (OpenMP, parallel sources)\n");
    std::printf("=======================================================\n\n");

    std::printf("Enter number of nodes: ");
    if (std::scanf("%d", &n) != 1 || n < 2) {
        std::printf("Invalid number of nodes.\n"); return 1;
    }
    std::printf("Enter the Delta Value: ");
    if (std::scanf("%u", &delta_in) != 1 || delta_in == 0u) {
        std::printf("Delta must be > 0.\n"); return 1;
    }
    Weight delta = (Weight)delta_in;

    int density = (n <= 100) ? 40 : 20;
    std::printf("\n--- Building random graph ---\n");
    std::printf("Nodes: %d   Density: %d%%   Seed: 42\n", n, density);

    Graph *g = graph_create(n);
    fill_random_edges(g, density, 42);
    int m = count_directed_edges(g);

    std::printf("Graph generation complete.\n");
    std::printf("Nodes: %d\n", n);
    std::printf("Directed edges stored: %d\n", m);
    std::printf("Undirected edges stored: %d\n", m / 2);
    std::printf("OpenMP threads: %d\n\n", omp_get_max_threads());

    CSR G;
    build_csr_split(g, delta, G);

    std::vector<std::vector<Weight>> dist_ds(n, std::vector<Weight>(n, INF_W));

    std::printf("Running Delta-Stepping APSP (OpenMP, parallel sources)...\n");
    double t0 = omp_get_wtime();
    #pragma omp parallel for schedule(dynamic, 1)
    for (int s = 0; s < n; s++) {
        std::vector<Weight> tent;
        seq_delta_step(n, G, s, delta, tent);
        dist_ds[s] = std::move(tent);
    }
    double t1 = omp_get_wtime();
    double runtime_ms = (t1 - t0) * 1000.0;

    std::printf("\n=======================================================\n");
    std::printf("       APSP RUNTIME (delta = %u)\n", delta_in);
    std::printf("=======================================================\n");
    std::printf(" %-32s | %-12s\n", "Algorithm", "Time (ms)");
    std::printf("-------------------------------------------------------\n");
    std::printf(" %-32s | %12.4f\n", "Delta-Stepping (OpenMP)", runtime_ms);

    graph_free(g);
    return 0;
}

Writing apsp_delta_step_omp.cpp


In [ ]:
!g++ -std=c++17 -O3 -fopenmp apsp_delta_step_omp.cpp -o apsp_delta_step_omp

## 3) CUDA - `apsp_delta_step_cuda.cu`

In [ ]:
%%writefile apsp_delta_step_cuda.cu
#include <cuda_runtime.h>
#include <limits.h>
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define INF_INT 1000000000

#define CUDA_CHECK(call)                                                       \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error %s:%d: %s\n",                          \
                    __FILE__, __LINE__, cudaGetErrorString(err));              \
            exit(1);                                                           \
        }                                                                      \
    } while (0)

typedef unsigned int Weight;
static const Weight INF_W = 0xFFFFFFFFu;

/* ================= GRAPH ================= */
typedef struct {
    int vertices;
    int **adj_matrix;
} Graph;

Graph* graph_create(int v) {
    Graph *g = (Graph *)malloc(sizeof(Graph));
    g->vertices = v;
    g->adj_matrix = (int **)malloc(v * sizeof(int *));
    for (int i = 0; i < v; i++) {
        g->adj_matrix[i] = (int *)malloc(v * sizeof(int));
        for (int j = 0; j < v; j++)
            g->adj_matrix[i][j] = (i == j) ? 0 : INF_INT;
    }
    return g;
}

void graph_add_edge(Graph *g, int s, int d, int w) {
    if (s != d && g->adj_matrix[s][d] > w) g->adj_matrix[s][d] = w;
}

void graph_free(Graph *g) {
    for (int i = 0; i < g->vertices; i++) free(g->adj_matrix[i]);
    free(g->adj_matrix);
    free(g);
}

void fill_random_edges(Graph *g, int density_pct, unsigned int seed) {
    srand(seed);
    int n = g->vertices;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
            if (i != j && (rand() % 100) < density_pct)
                graph_add_edge(g, i, j, (rand() % 100) + 1);
}

static int count_edges(Graph *g) {
    int n = g->vertices, m = 0;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
            if (i != j && g->adj_matrix[i][j] < INF_INT) m++;
    return m;
}

void matrix_free(int **m, int n) {
    for (int i = 0; i < n; i++) free(m[i]);
    free(m);
}

/* ================= EDGE LIST (light/heavy split) ================= */
typedef struct { int *eu; int *ev; Weight *ew; int n; } EdgeList;

static void free_edge_list(EdgeList *L) {
    free(L->eu); free(L->ev); free(L->ew);
}

/* ================= KERNELS ================= */
__global__ void k_init_dist(unsigned *d, int n, int src) {
    int v = blockIdx.x * blockDim.x + threadIdx.x;
    if (v >= n) return;
    d[v] = (v == src) ? 0u : INF_W;
}

__global__ void k_relax_edges(const int *eu, const int *ev, const Weight *ew, int ne,
                              const unsigned *dist_in, unsigned *dist_out,
                              const unsigned char *mask_u, Weight delta, int is_heavy) {
    int e = blockIdx.x * blockDim.x + threadIdx.x;
    if (e >= ne) return;
    int u = eu[e];
    int v = ev[e];
    Weight w = ew[e];
    if (is_heavy) { if (w <= delta) return; }
    else          { if (w >  delta) return; }
    if (!mask_u[u]) return;
    unsigned du = dist_in[u];
    if (du == INF_W) return;
    unsigned long long sum = (unsigned long long)du + (unsigned long long)w;
    if (sum > (unsigned long long)INF_W - 1ULL) return;
    atomicMin((unsigned *)&dist_out[v], (unsigned)sum);
}

/* ================= HOST BUCKET HELPERS ================= */
static int any_t_ge_i_delta(const unsigned *t, int n, unsigned long long i, Weight delta) {
    unsigned long long thr = i * (unsigned long long)delta;
    for (int v = 0; v < n; v++) {
        if (t[v] == INF_W) continue;
        if ((unsigned long long)t[v] >= thr) return 1;
    }
    return 0;
}

static void mask_in_bucket(const unsigned *t, int n, unsigned long long i, Weight delta,
                           unsigned char *mask) {
    unsigned long long lo = i * (unsigned long long)delta;
    unsigned long long hi = (i + 1ULL) * (unsigned long long)delta;
    for (int v = 0; v < n; v++) {
        if (t[v] == INF_W) { mask[v] = 0; continue; }
        unsigned long long tv = t[v];
        mask[v] = (tv >= lo && tv < hi) ? 1 : 0;
    }
}

static void mask_S_and_t(const unsigned *t, int n, const unsigned char *S, unsigned char *m) {
    for (int v = 0; v < n; v++)
        m[v] = (S[v] && t[v] != INF_W) ? 1 : 0;
}

/* Per-source CUDA delta-stepping using device-resident edge lists */
static void cuda_delta_step_one(int n, int src, Weight delta,
                                int *d_leu, int *d_lev, Weight *d_lew, int nL,
                                int *d_heu, int *d_hev, Weight *d_hew, int nH,
                                unsigned *d0, unsigned *d1, unsigned char *d_mask,
                                unsigned *h_t, unsigned *h_prev,
                                unsigned char *h_mask, unsigned char *h_S,
                                unsigned *snap, unsigned *h_out)
{
    int threads = 256;
    int blocks_n = (n + threads - 1) / threads;

    k_init_dist<<<blocks_n, threads>>>(d0, n, src);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(h_t, d0, sizeof(unsigned) * n, cudaMemcpyDeviceToHost));

    unsigned long long i_phase = 0;
    while (any_t_ge_i_delta(h_t, n, i_phase, delta)) {
        memset(h_S, 0, n);

        for (;;) {
            mask_in_bucket(h_t, n, i_phase, delta, h_mask);
            int any = 0;
            for (int k = 0; k < n; k++) if (h_mask[k]) { any = 1; break; }
            if (!any) break;

            for (int k = 0; k < n; k++) if (h_mask[k]) h_S[k] = 1;

            memcpy(snap, h_t, sizeof(unsigned) * n);
            memcpy(h_prev, h_t, sizeof(unsigned) * n);
            CUDA_CHECK(cudaMemcpy(d0, h_t, sizeof(unsigned) * n, cudaMemcpyHostToDevice));

            int inner_iters = 0;
            const int max_inner = n + 64;
            while (inner_iters < max_inner) {
                CUDA_CHECK(cudaMemcpy(d_mask, h_mask, n, cudaMemcpyHostToDevice));
                CUDA_CHECK(cudaMemcpy(d1, d0, sizeof(unsigned) * n, cudaMemcpyDeviceToDevice));
                if (nL > 0) {
                    int bl = (nL + threads - 1) / threads;
                    k_relax_edges<<<bl, threads>>>(d_leu, d_lev, d_lew, nL, d0, d1, d_mask, delta, 0);
                    CUDA_CHECK(cudaDeviceSynchronize());
                }
                CUDA_CHECK(cudaMemcpy(d0, d1, sizeof(unsigned) * n, cudaMemcpyDeviceToDevice));
                CUDA_CHECK(cudaMemcpy(h_t, d0, sizeof(unsigned) * n, cudaMemcpyDeviceToHost));

                int changed = 0;
                for (int k = 0; k < n; k++) if (h_t[k] != h_prev[k]) { changed = 1; break; }
                memcpy(h_prev, h_t, sizeof(unsigned) * n);
                mask_in_bucket(h_t, n, i_phase, delta, h_mask);
                inner_iters++;
                if (!changed) break;
            }

            int no_progress = 1;
            for (int k = 0; k < n; k++) if (h_t[k] != snap[k]) { no_progress = 0; break; }
            if (no_progress) break;
        }

        mask_S_and_t(h_t, n, h_S, h_mask);
        CUDA_CHECK(cudaMemcpy(d_mask, h_mask, n, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d0, h_t, sizeof(unsigned) * n, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d1, d0, sizeof(unsigned) * n, cudaMemcpyDeviceToDevice));
        if (nH > 0) {
            int bh = (nH + threads - 1) / threads;
            k_relax_edges<<<bh, threads>>>(d_heu, d_hev, d_hew, nH, d0, d1, d_mask, delta, 1);
            CUDA_CHECK(cudaDeviceSynchronize());
        }
        CUDA_CHECK(cudaMemcpy(d0, d1, sizeof(unsigned) * n, cudaMemcpyDeviceToDevice));
        CUDA_CHECK(cudaMemcpy(h_t, d0, sizeof(unsigned) * n, cudaMemcpyDeviceToHost));

        i_phase++;
    }

    memcpy(h_out, h_t, sizeof(unsigned) * n);
}

int main(void) {
    int n;
    unsigned int delta_in;

    printf("=======================================================\n");
    printf("  DELTA-STEPPING APSP (CUDA)\n");
    printf("=======================================================\n\n");

    int gpu_count = 0;
    if (cudaGetDeviceCount(&gpu_count) != cudaSuccess || gpu_count == 0) {
        fprintf(stderr, "No CUDA-capable GPU detected.\n"); return 1;
    }
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("CUDA Device: %s (Compute Capability %d.%d)\n\n",
           prop.name, prop.major, prop.minor);

    printf("Enter number of nodes: ");
    if (scanf("%d", &n) != 1 || n < 2) { printf("Invalid n.\n"); return 1; }
    printf("Enter the Delta Value: ");
    if (scanf("%u", &delta_in) != 1 || delta_in == 0u) { printf("Bad delta.\n"); return 1; }
    Weight delta = (Weight)delta_in;

    int density = (n <= 100) ? 40 : 20;
    printf("\n--- Building random graph ---\n");
    printf("Nodes: %d   Density: %d%%   Seed: 42\n", n, density);

    Graph *g = graph_create(n);
    fill_random_edges(g, density, 42);
    int m = count_edges(g);

    printf("Graph generation complete.\n");
    printf("Nodes: %d\n", n);
    printf("Directed edges stored: %d\n", m);
    printf("Undirected edges stored: %d\n\n", m / 2);

    /* Split into light/heavy edge lists on host */
    EdgeList L = {NULL, NULL, NULL, 0};
    EdgeList H = {NULL, NULL, NULL, 0};
    int cap = (m > 0) ? m : 1;
    L.eu = (int *)malloc(cap * sizeof(int));
    L.ev = (int *)malloc(cap * sizeof(int));
    L.ew = (Weight *)malloc(cap * sizeof(Weight));
    H.eu = (int *)malloc(cap * sizeof(int));
    H.ev = (int *)malloc(cap * sizeof(int));
    H.ew = (Weight *)malloc(cap * sizeof(Weight));
    for (int u = 0; u < n; u++) {
        for (int v = 0; v < n; v++) {
            if (u != v && g->adj_matrix[u][v] < INF_INT) {
                Weight w = (Weight)g->adj_matrix[u][v];
                if (w <= delta) {
                    L.eu[L.n] = u; L.ev[L.n] = v; L.ew[L.n] = w; L.n++;
                } else {
                    H.eu[H.n] = u; H.ev[H.n] = v; H.ew[H.n] = w; H.n++;
                }
            }
        }
    }

    /* Upload edges ONCE (reused across all sources) */
    int *d_leu, *d_lev; Weight *d_lew;
    int *d_heu, *d_hev; Weight *d_hew;
    int nL_cap = (L.n > 0) ? L.n : 1;
    int nH_cap = (H.n > 0) ? H.n : 1;
    CUDA_CHECK(cudaMalloc(&d_leu, sizeof(int)    * nL_cap));
    CUDA_CHECK(cudaMalloc(&d_lev, sizeof(int)    * nL_cap));
    CUDA_CHECK(cudaMalloc(&d_lew, sizeof(Weight) * nL_cap));
    CUDA_CHECK(cudaMalloc(&d_heu, sizeof(int)    * nH_cap));
    CUDA_CHECK(cudaMalloc(&d_hev, sizeof(int)    * nH_cap));
    CUDA_CHECK(cudaMalloc(&d_hew, sizeof(Weight) * nH_cap));
    if (L.n > 0) {
        CUDA_CHECK(cudaMemcpy(d_leu, L.eu, sizeof(int)    * L.n, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_lev, L.ev, sizeof(int)    * L.n, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_lew, L.ew, sizeof(Weight) * L.n, cudaMemcpyHostToDevice));
    }
    if (H.n > 0) {
        CUDA_CHECK(cudaMemcpy(d_heu, H.eu, sizeof(int)    * H.n, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_hev, H.ev, sizeof(int)    * H.n, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_hew, H.ew, sizeof(Weight) * H.n, cudaMemcpyHostToDevice));
    }

    /* Per-source working buffers (reused across sources) */
    unsigned *d0, *d1;
    unsigned char *d_mask;
    CUDA_CHECK(cudaMalloc(&d0,     sizeof(unsigned) * n));
    CUDA_CHECK(cudaMalloc(&d1,     sizeof(unsigned) * n));
    CUDA_CHECK(cudaMalloc(&d_mask, n));
    unsigned       *h_t    = (unsigned *)malloc(sizeof(unsigned) * n);
    unsigned       *h_prev = (unsigned *)malloc(sizeof(unsigned) * n);
    unsigned       *snap   = (unsigned *)malloc(sizeof(unsigned) * n);
    unsigned char  *h_mask = (unsigned char *)malloc(n);
    unsigned char  *h_S    = (unsigned char *)calloc(n, 1);

    /* APSP distance matrix */
    unsigned **dist_ds = (unsigned **)malloc(n * sizeof(unsigned *));
    for (int i = 0; i < n; i++)
        dist_ds[i] = (unsigned *)malloc(sizeof(unsigned) * n);

    /* Run delta-stepping from every source */
    printf("Running Delta-Stepping APSP (CUDA, per-source kernels)...\n");
    cudaEvent_t ev0, ev1;
    cudaEventCreate(&ev0);
    cudaEventCreate(&ev1);
    cudaEventRecord(ev0);

    for (int s = 0; s < n; s++) {
        cuda_delta_step_one(n, s, delta,
                            d_leu, d_lev, d_lew, L.n,
                            d_heu, d_hev, d_hew, H.n,
                            d0, d1, d_mask,
                            h_t, h_prev, h_mask, h_S, snap,
                            dist_ds[s]);
    }

    cudaEventRecord(ev1);
    cudaEventSynchronize(ev1);
    float ms;
    cudaEventElapsedTime(&ms, ev0, ev1);
    double runtime_ms = (double)ms;

    printf("\n=======================================================\n");
    printf("       APSP RUNTIME (delta = %u)\n", delta_in);
    printf("=======================================================\n");
    printf(" %-32s | %-12s\n", "Algorithm", "Time (ms)");
    printf("-------------------------------------------------------\n");
    printf(" %-32s | %12.4f\n", "Delta-Stepping (CUDA)", runtime_ms);

    for (int i = 0; i < n; i++) free(dist_ds[i]);
    free(dist_ds);
    free(h_t); free(h_prev); free(snap); free(h_mask); free(h_S);
    cudaFree(d0); cudaFree(d1); cudaFree(d_mask);
    cudaFree(d_leu); cudaFree(d_lev); cudaFree(d_lew);
    cudaFree(d_heu); cudaFree(d_hev); cudaFree(d_hew);
    free_edge_list(&L); free_edge_list(&H);
    graph_free(g);
    cudaEventDestroy(ev0); cudaEventDestroy(ev1);
    return 0;
}

Overwriting apsp_delta_step_cuda.cu


In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_75 apsp_delta_step_cuda.cu -o apsp_delta_step_cuda